# Initial Exploration

An initial exploration, for which the Jupyter notebook is better suited, is crucial to understand the structure and content of the datasets we are working with. This step allows us to identify the variables available, their types, and the relationships between them. It also helps us to detect any potential issues with the data, such as missing values or inconsistencies, which could affect our analysis.
The pipeline is particularly complex since it involves dealing with multiple datasets, each with its own structure and coding system and different variables, all of which have different codes which can be difficult to interpret, it can be hard to understand what they measure, if they can be used and compared together and if they are raw measures or simply indexes. Therefore, it is crucial to have a clear understanding of the datasets and the variables they contain before proceeding with the analysis. This involves exploring the datasets, understanding the meaning of the variables, and determining how they can be used in the context of our analysis.Also, the project involves analyzing growth accounts and intangibles data, which are categorized by NACE codes and IO tables, which instead use ICIO codes. Hence, a crucial step in our analysis is to map the NACE codes from the growth accounts to the ICIO codes used in the IO tables. This mapping process is complex and requires careful handling to ensure accurate analysis.

In [1]:
import pandas as pd
import os

# Set up paths
data_raw_path = os.path.join('..', 'data', 'raw')
print(f"Data raw path:{data_raw_path}")

Data raw path: ..\data\raw


## 1) Sector Codes Analysis

Load the growth account CSV file and the intangibles analytical files into a pandas DataFrame and display basic information about the dataset structure. In order to understand it, we visualize the first few rows of the dataset. Then, we print the unique sector codes.

In [7]:
# Load growth accounts CSV
growth_accounts_path = os.path.join(data_raw_path, 'growth accounts.csv')
df_growth = pd.read_csv(growth_accounts_path)

# Display basic information about the dataset structure
print("Growth Accounts Dataset Shape:", df_growth.shape)
print("The dataset contains the following columns:", df_growth.columns)

# Unique NACE codes in growth accounts
unique_nace = df_growth['nace_r2_code'].unique()
print(f"The dataset contains {len(unique_nace)} unique NACE codes.")
print("Unique NACE codes:", unique_nace)

print("\n" + "="*50 + "\n")

# Load intangibles analytical files
intangibles_path = os.path.join(data_raw_path, 'intangibles analytical.csv')
df_intangibles = pd.read_csv(intangibles_path)

# Display basic information about the intangibles dataset structure
print("Intangibles Dataset Shape:", df_intangibles.shape)
print("The intangibles dataset contains the following columns:", df_intangibles.columns)

# Unique NACE codes in intangibles dataset
unique_nace_intangibles = df_intangibles['nace_r2_code'].unique()
print(f"The intangibles dataset contains {len(unique_nace_intangibles)} unique NACE codes.")
print("Unique NACE codes in intangibles dataset:", unique_nace_intangibles)

print("\n" + "="*50 + "\n")

# Check if the sectors are the same in both datasets
if set(unique_nace) == set(unique_nace_intangibles):
    print("The NACE codes in both datasets match.")


Growth Accounts Dataset Shape: (2467044, 8)
The dataset contains the following columns: Index(['Unnamed: 0', 'nace_r2_code', 'geo_code', 'year', 'nace_r2_name',
       'geo_name', 'var', 'value'],
      dtype='object')
The dataset contains 58 unique NACE codes.
Unique NACE codes: ['A' 'B' 'C' 'C10-C12' 'C13-C15' 'C16-C18' 'C19' 'C20' 'C20-C21' 'C21'
 'C22-C23' 'C24-C25' 'C26' 'C26-C27' 'C27' 'C28' 'C29-C30' 'C31-C33' 'D'
 'D-E' 'E' 'F' 'G' 'G45' 'G46' 'G47' 'H' 'H49' 'H50' 'H51' 'H52' 'H53' 'I'
 'J' 'J58-J60' 'J61' 'J62-J63' 'K' 'L' 'L68A' 'M' 'M-N' 'MARKT' 'MARKTxAG'
 'N' 'O' 'O-Q' 'P' 'Q' 'Q86' 'Q87-Q88' 'R' 'R-S' 'S' 'T' 'TOT' 'TOT_IND'
 'U']


Intangibles Dataset Shape: (61074, 134)
The intangibles dataset contains the following columns: Index(['Unnamed: 0', 'nace_r2_code', 'geo_code', 'nace_r2_name', 'geo_name',
       'year', 'HK_Innovprop', 'HK_Intang', 'HK_NatAcc', 'HK_Tang',
       ...
       'Kq_TangNRes', 'Kq_Train', 'VA_CP', 'VA_PI', 'VA_PYP', 'VA_Q', 'VAadj',
       'VAadj

Load the intangibles analytical CSV file and extract unique sectors from it.

In [10]:
# Load IO tables CSV

Intangibles Analytical Dataset Shape: (61074, 134)
The intangibles analytical dataset contains 58 unique NACE codes.
Unique NACE codes in intangibles analytical dataset: ['A' 'B' 'C' 'C10-C12' 'C13-C15' 'C16-C18' 'C19' 'C20' 'C20-C21' 'C21'
 'C22-C23' 'C24-C25' 'C26' 'C26-C27' 'C27' 'C28' 'C29-C30' 'C31-C33' 'D'
 'D-E' 'E' 'F' 'G' 'G45' 'G46' 'G47' 'H' 'H49' 'H50' 'H51' 'H52' 'H53' 'I'
 'J' 'J58-J60' 'J61' 'J62-J63' 'K' 'L' 'L68A' 'M' 'M-N' 'MARKT' 'MARKTxAG'
 'N' 'O' 'O-Q' 'P' 'Q' 'Q86' 'Q87-Q88' 'R' 'R-S' 'S' 'T' 'TOT' 'TOT_IND'
 'U']


The first two datasets follow the same NACE convention, while the IO tables follow the ICIO convention. 

## 2) Variables Analysis (Italy Only)

This section gives a clear overview of the variables available in the EUKLEMS datasets for Italy (geo_code equals IT).

It shows the unique variables in Growth Accounts and Intangibles Analytical.
It also shows yearly summary statistics for each variable: missing values, mean, median, minimum, and maximum.

In [12]:
df_growth_it = df_growth[df_growth["geo_code"] == "IT"].copy()
df_intangibles_it = df_intangibles[df_intangibles["geo_code"] == "IT"].copy()

for _df in (df_growth_it, df_intangibles_it):
    _df["year"] = pd.to_numeric(_df["year"], errors="coerce")

print("Italy rows in growth accounts:", len(df_growth_it))
print("Italy rows in intangibles analytical:", len(df_intangibles_it))

growth_vars = sorted(df_growth_it["var"].dropna().astype(str).unique()) if "var" in df_growth_it.columns else []
id_cols = {"geo_code", "geo_name", "nace_r2_code", "nace_r2_name", "year"}
intangibles_vars = sorted(c for c in df_intangibles_it.columns if c not in id_cols)
all_vars = sorted(set(growth_vars).union(intangibles_vars))

print("\nGrowth Accounts unique variables (Italy):", len(growth_vars))
print(growth_vars)
print("\nIntangibles Analytical unique variables (Italy):", len(intangibles_vars))
print(intangibles_vars)
print("\nCombined unique variables across both datasets (Italy):", len(all_vars))
print(all_vars)

growth_stats = pd.DataFrame()
if {"year", "var", "value"}.issubset(df_growth_it.columns):
    g = df_growth_it[["year", "var", "value"]].copy()
    g["value"] = pd.to_numeric(g["value"], errors="coerce")
    growth_stats = (
        g.groupby(["year", "var"], dropna=False)["value"]
         .agg(
             missing_values=lambda s: s.isna().sum(),
             mean="mean",
             median="median",
             min="min",
             max="max",
         )
         .reset_index()
         .sort_values(["year", "var"])
    )

print("\nGrowth Accounts yearly stats (first 20 rows):")
print(growth_stats.to_string(index=False))

intangibles_stats = pd.DataFrame()
if intangibles_vars:
    i = df_intangibles_it[["year"] + intangibles_vars].copy()
    i[intangibles_vars] = i[intangibles_vars].apply(pd.to_numeric, errors="coerce")
    i_long = i.melt(id_vars=["year"], value_vars=intangibles_vars, var_name="var", value_name="value")
    intangibles_stats = (
        i_long.groupby(["year", "var"], dropna=False)["value"]
             .agg(
                 missing_values=lambda s: s.isna().sum(),
                 mean="mean",
                 median="median",
                 min="min",
                 max="max",
             )
             .reset_index()
             .sort_values(["year", "var"])
    )

print("\nIntangibles Analytical yearly stats (first 20 rows):")
print(intangibles_stats.head(20).to_string(index=False))

Italy rows in growth accounts: 73224
Italy rows in intangibles analytical: 1566

Growth Accounts unique variables (Italy): 68
['CAP', 'CAPEconComp_QI', 'CAPICT_QI', 'CAPInnovprop_QI', 'CAPIntang_QI', 'CAPNICT_QI', 'CAPTang_QI', 'CAP_QI', 'LAB', 'LAB_QI', 'LP1ConEconComp', 'LP1ConInnovprop', 'LP1ConIntang', 'LP1ConIntangNA', 'LP1ConIntang_BU', 'LP1ConIntangnonNA', 'LP1ConLC', 'LP1ConLC_BU', 'LP1ConTFP', 'LP1ConTFP_BU', 'LP1ConTangICT', 'LP1ConTangICT_BU', 'LP1ConTangNICT', 'LP1ConTangNICT_BU', 'LP1Con_Soft_DB', 'LP1TFP_BU_I', 'LP1TFP_I', 'LP1_BU_G', 'LP1_G', 'LP2ConEconComp', 'LP2ConIntang', 'LP2ConIntangNA', 'LP2ConIntang_BU', 'LP2ConIntangnonNA', 'LP2ConLC', 'LP2ConLC_BU', 'LP2ConTFP', 'LP2ConTFP_BU', 'LP2ConTangICT', 'LP2ConTangICT_BU', 'LP2ConTangNICT', 'LP2ConTangNICT_BU', 'LP2Con_Soft_DB', 'LP2TFP_BU_I', 'LP2TFP_I', 'LP2_BU_G', 'LP2_G', 'VAConEconComp', 'VAConH', 'VAConH_BU', 'VAConInnovprop', 'VAConIntang', 'VAConIntangNA', 'VAConIntang_BU', 'VAConIntangnonNA', 'VAConLC', 'VAConL